In [ ]:
from Tools_U1_plaquette import *

In [ ]:
# ---------------- Parameters ----------------
npoints = 30
λ_ovr_g = np.logspace(-1, 3, num=npoints)
λ_ovr_m = np.logspace(-1, 3, num=npoints)

N = 13
N_θ = 10001

# ---------------- Storage ----------------
expval_plaquette_operator = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))

expval_n12 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_n12_sqr = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_variance_n12 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))

expval_n24 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_n24_sqr = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_variance_n24 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))

expval_n13 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_n13_sqr = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_variance_n13 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))

expval_n34 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_n34_sqr = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))
expval_variance_n34 = np.zeros((len(λ_ovr_g), len(λ_ovr_m)))

# ---------------- Loop ----------------
for i, λ_g in enumerate(λ_ovr_g):
    for j, λ_m in enumerate(λ_ovr_m):

        print("Computing for λ/g =", λ_g, ", λ/m =", λ_m)
        (
            E_ν_vector, 
            exp_plusiθ_op_matrix, 
            exp_minusiθ_op_matrix, 
            nθ_op_matrix, 
            nθ_sqr_op_matrix
        ) = single_link_data(λ_m, λ_g, N, N_θ)
        """
        H = H_0 + H_additional

        H_0 = Σ_l H_l, with H_l = Σ_l [ 2/(λ/m) + 1/(λ/g) ] n_l^2 - cos(θ_l), l={12,24,13,34} 
        have analytical solution obdatied in single_link_data(), so in Mathieu basis are 
        diagonal with same eigenvalues E_ν_vector for each link.

        H_additional = 2/(λ/m) * (n12 * n13 + n24 * n34 - n12 * n24 - n13 * n34 )
        """
        H_0_l = sp.diags(E_ν_vector, offsets = 0, format='csr')
        id_N = sp.eye(N, format='csr')
        H_0 = (
              kron_n(H_0_l, id_N, id_N, id_N) 
            + kron_n(id_N, H_0_l, id_N, id_N) 
            + kron_n(id_N, id_N, H_0_l, id_N) 
            + kron_n(id_N, id_N, id_N, H_0_l)
        )

        Udiff = exp_plusiθ_op_matrix.conj().T - exp_minusiθ_op_matrix
        print("exponentials are nice? - Error = ", np.max(np.abs(Udiff)))
        D = nθ_op_matrix - nθ_op_matrix.conj().T
        print("single-link max asymmetry:", np.max(np.abs(D)))
        D2 = nθ_sqr_op_matrix - nθ_sqr_op_matrix.conj().T
        print("single-link max asymmetry:", np.max(np.abs(D2)))
        
        # Choice of order is important for Kronecker products. Here we take [n12, n24, n13, n34]
        H_additional = (2.0/λ_m) * (
            kron_n(nθ_op_matrix, id_N, nθ_op_matrix, id_N) # n12 * n13
          + kron_n(id_N, nθ_op_matrix, id_N, nθ_op_matrix) # n24 * n34
          - kron_n(nθ_op_matrix, nθ_op_matrix, id_N, id_N) # n12 * n24
          - kron_n(id_N, id_N, nθ_op_matrix, nθ_op_matrix) # n13 * n34
        )
        H_mathieu = H_0 + H_additional

        is_hermitian = (H_mathieu - H_mathieu.getH()).nnz == 0
        print("Hermitian?", is_hermitian)
        assert sp.isspmatrix(H_mathieu)
        off_diag_nnz_mathieu = (H_mathieu - sp.diags(H_mathieu.diagonal(), format="csr")).nnz
        print("Off-diagonal elements:", off_diag_nnz_mathieu)
        print("Shape of the Hamiltonian H_mathieu -> ", H_mathieu.shape)

        E, ψ = spla.eigsh(H_mathieu, k=1, which="SA", tol=1e-8, maxiter=100000)
        ψ_chosen = ψ[:, 0]

        P = 0.5 * (
            kron_n(
                exp_plusiθ_op_matrix, 
                exp_plusiθ_op_matrix, 
                exp_minusiθ_op_matrix, 
                exp_minusiθ_op_matrix
            ) + kron_n(
                exp_minusiθ_op_matrix, 
                exp_minusiθ_op_matrix, 
                exp_plusiθ_op_matrix, 
                exp_plusiθ_op_matrix
            )
        )

        n12 = kron_n(nθ_op_matrix, id_N, id_N, id_N)
        n12_sqr = kron_n(nθ_sqr_op_matrix, id_N, id_N, id_N)
        n24 = kron_n(id_N, nθ_op_matrix, id_N, id_N)
        n24_sqr = kron_n(id_N, nθ_sqr_op_matrix, id_N, id_N)
        n13 = kron_n(id_N, id_N, nθ_op_matrix, id_N)
        n13_sqr = kron_n(id_N, id_N, nθ_sqr_op_matrix, id_N)
        n34 = kron_n(id_N, id_N, id_N, nθ_op_matrix)
        n34_sqr = kron_n(id_N, id_N, id_N, nθ_sqr_op_matrix)

        expval_plaquette_operator[i, j] =  np.vdot(ψ_chosen, P @ ψ_chosen).real
        expval_n12[i, j] =  np.vdot(ψ_chosen, n12 @ ψ_chosen).real
        expval_n12_sqr[i, j] =  np.vdot(ψ_chosen, n12_sqr @ ψ_chosen).real
        expval_variance_n12[i, j] =  (
              np.vdot(ψ_chosen, n12_sqr @ ψ_chosen).real 
            - np.vdot(ψ_chosen, n12 @ ψ_chosen).real**2
        )
        expval_n24[i, j] =  np.vdot(ψ_chosen, n24 @ ψ_chosen).real
        expval_n24_sqr[i, j] =  np.vdot(ψ_chosen, n24_sqr @ ψ_chosen).real
        expval_variance_n24[i, j] =  (
              np.vdot(ψ_chosen, n24_sqr @ ψ_chosen).real 
            - np.vdot(ψ_chosen, n24 @ ψ_chosen).real**2
        )
        expval_n13[i, j] =  np.vdot(ψ_chosen, n13 @ ψ_chosen).real
        expval_n13_sqr[i, j] =  np.vdot(ψ_chosen, n13_sqr @ ψ_chosen).real
        expval_variance_n13[i, j] =  (
              np.vdot(ψ_chosen, n13_sqr @ ψ_chosen).real 
            - np.vdot(ψ_chosen, n13 @ ψ_chosen).real**2
        )
        expval_n34[i, j] =  np.vdot(ψ_chosen, n34 @ ψ_chosen).real
        expval_n34_sqr[i, j] =  np.vdot(ψ_chosen, n34_sqr @ ψ_chosen).real
        expval_variance_n34[i, j] =  (
              np.vdot(ψ_chosen, n34_sqr @ ψ_chosen).real 
            - np.vdot(ψ_chosen, n34 @ ψ_chosen).real**2
        )

        print(f"λ/m={λ_m:.6e}, λ/g={λ_g:.6e} → < Pop > ={expval_plaquette_operator[i, j]:.7f}\n")
        print(f"< n12 > ={expval_n12[i, j]:.7f}, < n12^2 > ={expval_n12_sqr[i, j]:.7f}, Var(n12)={expval_variance_n12[i, j]:.7f}\n")
        print(f"< n24 > ={expval_n24[i, j]:.7f}, < n24^2 > ={expval_n24_sqr[i, j]:.7f}, Var(n24)={expval_variance_n24[i, j]:.7f}\n")
        print(f"< n13 > ={expval_n13[i, j]:.7f}, < n13^2 > ={expval_n13_sqr[i, j]:.7f}, Var(n13)={expval_variance_n13[i, j]:.7f}\n")
        print(f"< n34 > ={expval_n34[i, j]:.7f}, < n34^2 > ={expval_n34_sqr[i, j]:.7f}, Var(n34)={expval_variance_n34[i, j]:.7f}\n")

In [ ]:
# Save results
filename = (
    f"plaquette_reduced4vars_alphas0_"
    f"expvals_data_meshgrids_N_{N}_npoints_{npoints}_mathieu_basis_sparse.npz"
)
np.savez_compressed(filename,
                    lambda_over_gs=λ_ovr_g,
                    lambda_over_ms=λ_ovr_m,
                    expvals_plaquette_operator=expval_plaquette_operator,
                    expvals_n12=expval_n12,
                    expvals_n12_sqr=expval_n12_sqr,
                    expvals_variance_n12=expval_variance_n12,
                    expvals_n24=expval_n24,
                    expvals_n24_sqr=expval_n24_sqr,
                    expvals_variance_n24=expval_variance_n24,
                    expvals_n13=expval_n13,
                    expvals_n13_sqr=expval_n13_sqr,
                    expvals_variance_n13=expval_variance_n13,
                    expvals_n34=expval_n34,
                    expvals_n34_sqr=expval_n34_sqr,
                    expvals_variance_n34=expval_variance_n34,
                    N=N)
print(f"\nResults saved to {filename}")